In [ ]:
import json, os, faiss, numpy as np
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# ===== CONFIG =====
CHUNKS = Path("processed/chunks.jsonl")
OUT_DIR = Path("indexes")
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cuda"  # "cpu" nếu không có GPU
BATCH_SIZE = 48  # Nitro5 GPU ~4GB → tối đa 48
INDEX_TYPE = "hnsw"
# ==================

OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_FILE = OUT_DIR / "faiss_hnsw.index"
ID_MAP_FILE = OUT_DIR / "id_map.json"

def stream_chunks(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

model = SentenceTransformer(MODEL, device=DEVICE)
dummy = model.encode(["test"], convert_to_numpy=True)
dim = dummy.shape[1]
index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 200

id_list = []
batch_texts = []

for item in tqdm(stream_chunks(CHUNKS), desc="Embedding chunks"):
    batch_texts.append(item["text"])
    id_list.append(item["id"])
    if len(batch_texts) >= BATCH_SIZE:
        embs = model.encode(batch_texts, convert_to_numpy=True, show_progress_bar=False).astype("float32")
        faiss.normalize_L2(embs)
        index.add(embs)
        batch_texts = []

if batch_texts:
    embs = model.encode(batch_texts, convert_to_numpy=True, show_progress_bar=False).astype("float32")
    faiss.normalize_L2(embs)
    index.add(embs)

faiss.write_index(index, str(INDEX_FILE))
json.dump(id_list, open(ID_MAP_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"Index built: {INDEX_FILE}\nTotal vectors: {index.ntotal}")


ModuleNotFoundError: No module named 'faiss'

In [1]:
import sys, platform, struct
print("python:", sys.executable)
print("version:", sys.version)
print("platform:", platform.platform())
print("bits:", struct.calcsize("P")*8)

# dán vào ô notebook hoặc chạy trong Anaconda Prompt bằng: python -c "..."
import sys, subprocess
print("python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "show", "faiss-cpu"], check=False)

import sys, faiss, numpy as np
from sentence_transformers import SentenceTransformer
print(sys.executable, np.__version__, faiss.__version__, SentenceTransformer)


python: c:\Users\tuand\anaconda3\python.exe
version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
platform: Windows-11-10.0.26100-SP0
bits: 64
python: c:\Users\tuand\anaconda3\python.exe


ModuleNotFoundError: No module named 'faiss'